# G24 — Does appraisal of the conversation content forecast *which* negative emotion B reacts with? (5-fold CV, train+val; test untouched)

**Why.** Forecasters succeed mostly when B mirrors A (F14: accuracy 56.9% vs 27.7% on shifts) and hardly separate
angry / sad / disgust (G19, G23: NEG 0.56 from clips I–III vs 0.69 at clip IV, descriptive). Appraisal theory
predicts that these emotions differ in how the responder appraises the event: anger — caused by another person and
controllable; sadness — loss, low control; disgust — norm violation. That information is in *what was said*, which our
CLIP text features do not capture (G13, G20), and an LLM reading of the subtitles was never run (G1 not executed).

**Appraisal features (no labels).** An LLM reads subtitle lines I–III only (clip IV text is never sent) and rates, for
the person who will respond next, 10 appraisal dimensions in [0, 1] (valence in [−1, 1]): event valence, caused by
another person, caused by the responder, caused by circumstances, responder's control, loss, norm violation,
unexpectedness, goal obstruction, threat. A separate call asks for a direct forecast of the responder's emotion (control
arm). Responses are cached. Both calls are zero-shot and see no Hi-EF label.

**Arms** (same LR, folds, C selection and features as G23 `FC`): `FC` (CLIP/AudioSet features of I–III), `FC+app`,
`APP` (appraisal only), `FC+llm` (+ the LLM's direct 7-way forecast; control).

**Measure.** Pair AUC as G23; `NEG` = mean over angry/sad, angry/disgust, disgust/sad. **Shift subset** = MCIS whose
B label differs from A's clip-III label (gold labels used only to define the subset, never as inputs). 95% CI by a
bootstrap over source folders, paired across arms.

**Decision (fixed before running).**
* **PASS** if Δ_shift = NEG_shift(`FC+app`) − NEG_shift(`FC`) has CI lower bound > 0 **and** NEG_shift(`APP`) has CI
  lower bound > 0.5. Then appraisal read from the content carries shift-direction information that the current
  features lack → a method pilot on reaction (non-mirroring) forecasting is justified.
* Otherwise **STOP**: with this reading of the content, the direction of B's negative reaction is not forecastable
  from clips I–III.

Descriptive: NEG on all MCIS, UAR, `FC+llm` vs `FC+app` (whether a gain is specific to appraisal or to any LLM reading
of the text), and a contamination probe (does the LLM name the series?). Hi-EF comes from TV shows the LLM may know;
a PASS must be read together with that probe.

In [ ]:
!pip install -q openai

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456]                       # as G8b / G11 / G12
LR, WEIGHT_DECAY = 1e-4, 1e-5                # B1 settings (only needed by the shared model cell)
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)     # G8b, unchanged
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
BASE = dict(readout='query', nq=1, q_readonly=False, role_emb=True, clip_emb=True, absent='token',
            drop=(), speech_parts=('text', 'audio', 'voice'), clips=(0, 1, 2), seed_offset=0)
ARMS = []                       # no neural arms in G23
EXPERIMENTS = []
N_BOOT, PCA_K = 1000, 32
LR_CS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
NEG_PAIRS = [('angry', 'sad'), ('angry', 'disgust'), ('disgust', 'sad')]
VAL_PAIRS = [('angry', 'happy'), ('happy', 'sad'), ('disgust', 'happy')]

# ---- G24: LLM settings ----
MODEL = "gpt-4o-mini"          # any OpenAI chat model with JSON output
TEMPERATURE = 0.0              # None for models that reject the parameter
MAX_WORKERS = 8
CACHE_APP = f"{OUT_DIR}/g24_appraisal_{MODEL.replace('/', '_')}.jsonl"
CACHE_DIR = f"{OUT_DIR}/g24_direct_{MODEL.replace('/', '_')}.jsonl"

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## LLM reading of subtitles I–III (appraisal and, separately, a direct forecast)

In [ ]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("OPENAI_API_KEY", UserSecretsClient().get_secret("OPENAI_API_KEY"))
except Exception:
    pass
client = OpenAI()
APP_DIMS = ['event_valence', 'caused_by_other', 'caused_by_responder', 'caused_by_circumstance',
            'responder_control', 'loss', 'norm_violation', 'unexpected', 'goal_obstruction', 'threat']
SYSTEM = ("You are an expert in appraisal theory of emotion, annotating TV drama dialogue. You only see subtitle "
          "text, which may contain transcription errors.")
CONTEXT = '''Three consecutive subtitle lines from a conversation in a TV drama. Lines [1] and [2] are earlier context
(speakers not given). Line [3] is spoken by person A. The next line (not shown) will be spoken by a different person,
the RESPONDER, who reacts to what has happened.
[1] {t1}
[2] {t2}
[3] (A) {t3}
'''
TASK_APP = CONTEXT + '''
Rate how the RESPONDER most likely appraises the situation at this moment. Do not name an emotion.
- event_valence: how good (+1) or bad (-1) the situation is for the responder
- caused_by_other: caused by A or another person (0-1)
- caused_by_responder: caused by the responder themselves (0-1)
- caused_by_circumstance: caused by circumstances / nobody (0-1)
- responder_control: how much the responder can still change or cope with it (0-1)
- loss: something valued is lost or irreversibly damaged (0-1)
- norm_violation: something offensive, immoral, or repulsive happened (0-1)
- unexpected: how unexpected it is for the responder (0-1)
- goal_obstruction: it blocks something the responder wants (0-1)
- threat: it is a danger or threat to the responder (0-1)
Return JSON only with these 10 keys and numeric values.'''
TASK_DIR = CONTEXT + '''
Which emotion will the RESPONDER most likely express in the next line?
Labels: angry, disgust, fear, happy, neutral, sad, surprise.
Return JSON only, a probability for every label (summing to 1): {{"angry": p, ...}}'''


def call(task, row, retries=5):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": task.format(t1=row['t1'] or '(no text)', t2=row['t2'] or '(no text)',
                                                    t3=row['t3'] or '(no text)')}]
    kw = dict(model=MODEL, messages=msgs, response_format={"type": "json_object"})
    if TEMPERATURE is not None:
        kw["temperature"] = TEMPERATURE
    for attempt in range(retries):
        try:
            return json.loads(client.chat.completions.create(**kw).choices[0].message.content)
        except Exception as e:
            ERRORS.append(f"{type(e).__name__}: {e}")
            if "temperature" in str(e) and "temperature" in kw:
                kw.pop("temperature"); continue
            time.sleep(2 ** attempt)
    return None


def run_cache(task, path):
    cache = {}
    if os.path.exists(path):
        for line in open(path):
            d = json.loads(line); cache[d['sample_id']] = d['response']
    todo = [r for r in DEV.to_dict('records') if r['sample_id'] not in cache]
    print(f"{os.path.basename(path)}: cached {len(cache)} | to query {len(todo)}", flush=True)
    with ThreadPoolExecutor(MAX_WORKERS) as pool, open(path, 'a') as f:
        futs = {pool.submit(call, task, r): r['sample_id'] for r in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=os.path.basename(path), mininterval=10):
            sid, resp = futs[fut], fut.result()
            if resp is not None:
                cache[sid] = resp; f.write(json.dumps({'sample_id': sid, 'response': resp}) + "\n")
    n_ok = sum(s in cache for s in DEV.sample_id)
    print(f"  responses {n_ok}/{N}")
    if n_ok < 0.95 * N:
        raise RuntimeError("More than 5% of items have no response; re-run this cell (cached items are skipped).")
    return cache


ERRORS = []
if call(TASK_APP, DEV.iloc[0].to_dict(), retries=2) is None:
    raise RuntimeError(f"Pre-flight call failed (Internet on? OPENAI_API_KEY secret? MODEL?). {ERRORS[-1:]}")
C_APP, C_DIR = run_cache(TASK_APP, CACHE_APP), run_cache(TASK_DIR, CACHE_DIR)
if ERRORS:
    print(f"{len(ERRORS)} API errors, e.g.", sorted(set(ERRORS))[:2])


def num(d, k, lo, hi):
    try:
        return float(np.clip(float(d[k]), lo, hi))
    except Exception:
        return np.nan


APP = np.array([[num(C_APP.get(s) or {}, k, -1 if k == 'event_valence' else 0, 1) for k in APP_DIMS]
                for s in DEV.sample_id], np.float32)
miss = np.isnan(APP)
print("missing appraisal values per dimension:", dict(zip(APP_DIMS, miss.sum(0))))
if (miss.mean(0) > 0.05).any():
    raise RuntimeError("An appraisal dimension is missing in more than 5% of responses; check the JSON keys.")
APP = np.where(miss, np.nanmean(APP, 0), APP)         # rare; filled with the column mean (no labels involved)


def to_probs(d):
    d = {str(k).strip().lower(): v for k, v in d.items()} if isinstance(d, dict) else {}
    p = np.array([num(d, e, 0, 1) if e in d else 0.0 for e in EMO]); p = np.nan_to_num(p)
    return p / p.sum() if p.sum() > 0 else np.full(7, 1 / 7)


LLMP = np.stack([to_probs(C_DIR.get(s)) for s in DEV.sample_id]).astype(np.float32)
print(pd.DataFrame(APP, columns=APP_DIMS).describe().loc[['mean', 'std']].round(2).to_string())

## Logistic regressions (same set-up as G23 `FC`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as _PCA

sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
print("fold sizes (MCIS):", load_)

ROWS = np.arange(N)                            # clip IV is not used in G24
yB_all, yA_all, src = DEV.yB.values, DEV.yA.values, DEV.source_folder.values


def clip_blocks(clips):
    out = {k: np.zeros((len(clips), d), np.float32) for k, d in (('face', 512), ('scene', 512), ('text', 512),
                                                                 ('audio', 527), ('flags', 2))}
    ok = np.array([c in CIDX for c in clips])
    ix = torch.tensor([CIDX[c] for c in np.asarray(clips)[ok]], device=DEVICE)
    fm = FEAT['fmask'][ix].unsqueeze(-1).float()
    af = FEAT['afound'][ix].float().unsqueeze(-1)
    out['face'][ok] = ((FEAT['face'][ix] * fm).sum(1) / fm.sum(1).clamp(min=1)).cpu().numpy()
    out['scene'][ok] = FEAT['ori'][ix].mean(1).cpu().numpy()
    out['text'][ok] = FEAT['text'][ix].cpu().numpy()
    out['audio'][ok] = (F.normalize(FEAT['audio'][ix], dim=-1) * af).cpu().numpy()
    out['flags'][ok] = torch.cat([af, (fm.sum(1) > 0).float()], 1).cpu().numpy()
    return out


BLK = {k: clip_blocks(DEV[f'clip{k}'].values) for k in (1, 2, 3)}
EXPR = np.zeros((N, 3, 11), np.float32)
for n, row in enumerate(DEV.itertuples()):
    for k, c in enumerate((row.clip1, row.clip2, row.clip3)):
        if len(FB[c]):
            EXPR[n, k, :10] = FB[c][:, :10].mean(0); EXPR[n, k, 10] = 1.0

V2 = ['face', 'scene', 'text', 'audio', 'flags']
FCSPEC = [(k, m) for k in (1, 2, 3) for m in V2]
TASKS = {
    'FC':       (FCSPEC, yB_all),
    'FC+app':   (FCSPEC + [(0, 'app')], yB_all),
    'APP':      ([(0, 'app')], yB_all),
    'FC+llm':   (FCSPEC + [(0, 'llm')], yB_all),
}


def design(spec, tr, te):
    # all PCAs are fitted on the training rows tr only
    A_, B_ = [], []
    for pos, mod in spec:
        if mod == 'expr':
            M = EXPR[:, pos - 1]
        elif mod == 'app':
            M = APP
        elif mod == 'llm':
            M = LLMP
        else:
            M = BLK[pos][mod]
        if mod in ('flags', 'expr', 'app', 'llm'):
            A_.append(M[tr]); B_.append(M[te]); continue
        fit = tr[BLK[pos]['flags'][tr, 0] > 0] if mod == 'audio' else tr
        p = _PCA(min(PCA_K, len(fit) - 1, M.shape[1]), random_state=0).fit(M[fit])
        a, b = p.transform(M[tr]), p.transform(M[te])
        if mod == 'audio':
            a, b = a * BLK[pos]['flags'][tr, :1], b * BLK[pos]['flags'][te, :1]
        A_.append(a); B_.append(b)
    return np.concatenate(A_, 1), np.concatenate(B_, 1)


def softmax_np(L):
    L = L - L.max(1, keepdims=True); E_ = np.exp(L); return E_ / E_.sum(1, keepdims=True)


def nll(L, y):
    return float(-np.log(softmax_np(L)[np.arange(len(y)), y] + 1e-12).mean())


def lr_logits(Xtr, ytr, Xte, C):
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(C=C, max_iter=3000).fit(sc.transform(Xtr), ytr)
    L = np.full((len(Xte), 7), np.nan)
    L[:, m.classes_] = m.decision_function(sc.transform(Xte))
    return np.where(np.isnan(L), np.nanmin(L, 1, keepdims=True) - 10, L)


LOGIT = {t: np.full((N, 7), np.nan) for t in TASKS}
lr_log = []
t0 = time.time()
for f in range(N_OUTER):
    tr = ROWS[fold_of_row[ROWS] != f]
    te = ROWS[fold_of_row[ROWS] == f]
    for t, (spec, yy) in TASKS.items():
        inner = {C: np.zeros((len(tr), 7)) for C in LR_CS}
        for ia, ib in GroupKFold(5).split(tr, groups=src[tr]):
            Xa, Xb = design(spec, tr[ia], tr[ib])
            for C in LR_CS:
                inner[C][ib] = lr_logits(Xa, yy[tr[ia]], Xb, C)
        scores = {C: nll(inner[C], yy[tr]) for C in LR_CS}
        C = min(scores, key=scores.get)
        Xa, Xb = design(spec, tr, te)
        LOGIT[t][te] = lr_logits(Xa, yy[tr], Xb, C)
        lr_log.append({'fold': f, 'task': t, 'C': C, 'n_features': Xa.shape[1], 'inner_nll': scores[C]})
    print(f"fold {f} done | {(time.time() - t0) / 60:.1f} min", flush=True)
assert all(not np.isnan(LOGIT[t][ROWS]).any() for t in TASKS)
pd.DataFrame(lr_log).to_csv(f"{OUT_DIR}/g24_lr_log.csv", index=False)
np.savez(f"{OUT_DIR}/g24_oof_logits.npz", sample_id=DEV.sample_id.values, fold=fold_of_row, yB=yB_all, yA=yA_all,
         src=src, rows=ROWS, **{t.replace('+', '_').replace('-', '_'): v for t, v in LOGIT.items()})
print("saved g24_oof_logits.npz and g24_lr_log.csv")

## Pair AUCs on all MCIS and on the shift subset; fixed decision

In [ ]:
from sklearn.metrics import roc_auc_score

E2 = {e: i for i, e in enumerate(EMO)}
SHIFT = yB_all != yA_all
print(f"shift subset: {SHIFT.sum()} / {N} MCIS")


def pair_auc(t, a, b, idx):
    m = idx[np.isin(yB_all[idx], [E2[a], E2[b]])]
    if len(np.unique(yB_all[m])) < 2:
        return np.nan
    return roc_auc_score(yB_all[m] == E2[a], LOGIT[t][m, E2[a]] - LOGIT[t][m, E2[b]])


def score(t, idx):
    d = {f"{a}/{b}": pair_auc(t, a, b, idx) for a, b in NEG_PAIRS}
    d['NEG'] = np.nanmean(list(d.values()))
    return d


groups = [np.where(src == e)[0] for e in np.unique(src)]
rng = np.random.default_rng(0)
DRAWS = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]) for _ in range(N_BOOT)]
SUB = {'all': lambda idx: idx, 'shift': lambda idx: idx[SHIFT[idx]]}
PT = {(t, s): score(t, f(np.arange(N))) for t in TASKS for s, f in SUB.items()}
BS = {(t, s): [score(t, f(idx)) for idx in DRAWS] for t in TASKS for s, f in SUB.items()}

rows = []
for (t, s), d in PT.items():
    r = {'task': t, 'subset': s, **d}
    r['NEG_lo'], r['NEG_hi'] = np.nanpercentile([b['NEG'] for b in BS[(t, s)]], [2.5, 97.5])
    m = np.arange(N) if s == 'all' else np.where(SHIFT)[0]
    r['UAR'] = war_uar(LOGIT[t][m].argmax(1), yB_all[m], 7)[1]
    rows.append(r)
R = pd.DataFrame(rows)
R.to_csv(f"{OUT_DIR}/g24_pair_auc.csv", index=False)
print(R.round(3).to_string(index=False))


def delta(a, b, s):
    pt = PT[(a, s)]['NEG'] - PT[(b, s)]['NEG']
    lo, hi = np.nanpercentile([x['NEG'] - y['NEG'] for x, y in zip(BS[(a, s)], BS[(b, s)])], [2.5, 97.5])
    return pt, lo, hi


d_shift = delta('FC+app', 'FC', 'shift')
app_lo = R[(R.task == 'APP') & (R.subset == 'shift')].NEG_lo.iloc[0]
print(f"\nΔ_shift = NEG_shift(FC+app) − NEG_shift(FC) = {d_shift[0]:+.3f} [{d_shift[1]:+.3f}, {d_shift[2]:+.3f}]")
print(f"NEG_shift(APP) lower bound = {app_lo:.3f}")
for a, b, s in [('FC+app', 'FC', 'all'), ('FC+llm', 'FC', 'shift'), ('FC+app', 'FC+llm', 'shift')]:
    p, lo, hi = delta(a, b, s)
    print(f"  (descriptive) {a} − {b} [{s}]: {p:+.3f} [{lo:+.3f}, {hi:+.3f}]")
dec = 'PASS' if d_shift[1] > 0 and app_lo > 0.5 else 'STOP'
print(f"== G24 decision (fixed rule): {dec} ==")
json.dump({'decision': dec, 'delta_shift': list(map(float, d_shift)), 'NEG_shift_APP_lo': float(app_lo),
           'model': MODEL}, open(f"{OUT_DIR}/g24_decision.json", 'w'), indent=1)

## Contamination probe (for the limitations section)

In [ ]:
probe = DEV.sample(n=30, random_state=0)
names = []
for r in probe.to_dict('records'):
    q = (f"Which TV series are these subtitle lines from? Answer with the series name only, or 'unknown'.\n"
         f"[1] {r['t1']}\n[2] {r['t2']}\n[3] {r['t3']}")
    kw = dict(model=MODEL, messages=[{"role": "user", "content": q}])
    if TEMPERATURE is not None:
        kw["temperature"] = TEMPERATURE
    try:
        names.append(client.chat.completions.create(**kw).choices[0].message.content.strip())
    except Exception as e:
        names.append(f"error: {type(e).__name__}")
print(pd.Series(names).value_counts().head(10).to_string())
json.dump(names, open(f"{OUT_DIR}/g24_contamination_probe.json", 'w'), indent=1)